In [1]:
# #collect.py
# import requests
# import pandas as pd
# import time
#
# # ── Part A: Get list of all active mutual funds ──────────────────────────
# def get_all_funds():
#     """
#     mfapi.in is a free Indian API — no key needed.
#     Returns a list of every fund with its schemeCode and schemeName.
#     """
#     url = "https://api.mfapi.in/mf"
#     response = requests.get(url)
#     funds = response.json()           # This gives you a Python list
#     df = pd.DataFrame(funds)          # Turn it into a table
#     df.to_csv("all_funds.csv", index=False)
#     print(f"Found {len(df)} funds")
#     return df
#
# # ── Part B: Get NAV history for ONE fund ─────────────────────────────────
# def get_nav_history(scheme_code):
#     """
#     scheme_code is a number like 120503 (Axis Bluechip Fund).
#     The API returns the full NAV history — every trading day.
#     """
#     url = f"https://api.mfapi.in/mf/{scheme_code}"
#     response = requests.get(url)
#     data = response.json()
#
#     df = pd.DataFrame(data['data'])           # The 'data' key has the history
#     df['date'] = pd.to_datetime(df['date'], format='%d-%m-%Y')
#     df['nav']  = df['nav'].astype(float)
#     df['scheme_code'] = scheme_code           # Tag it so you know which fund
#     df = df.sort_values('date')               # Oldest date first
#     return df
#
# # ── Part C: Loop through your chosen funds & save ─────────────────────────
# def collect_sample_funds():
#     # Start small — just 5 popular funds by their scheme codes
#     sample_funds = {
#         120503: "Axis Bluechip Fund",
#         100033: "HDFC Top 100 Fund",
#         119598: "Mirae Asset Large Cap",
#         125354: "Parag Parikh Flexi Cap",
#         120465: "SBI Small Cap Fund",
#     }
#
#     all_nav = []
#
#     for code, name in sample_funds.items():
#         print(f"Fetching: {name}...")
#         nav_df = get_nav_history(code)
#         nav_df['fund_name'] = name
#         all_nav.append(nav_df)
#         time.sleep(0.5)     # Wait half a second — be polite to the API
#
#     combined = pd.concat(all_nav, ignore_index=True)
#     combined.to_csv("nav_history_raw.csv", index=False)
#     print(f"Saved {len(combined)} rows to nav_history_raw.csv")
#
# # ── Run it ────────────────────────────────────────────────────────────────
# if __name__ == "__main__":
#     get_all_funds()
#     collect_sample_funds()

Found 37597 funds
Fetching: Axis Bluechip Fund...
Fetching: HDFC Top 100 Fund...
Fetching: Mirae Asset Large Cap...
Fetching: Parag Parikh Flexi Cap...
Fetching: SBI Small Cap Fund...
Saved 17895 rows to nav_history_raw.csv


In [1]:
"""
collect.py — Data Collection Layer
===================================
Mutual Fund Analysis Dashboard
--------------------------------
This script is Stage 1 of the pipeline. Its only job is to fetch raw data
from external sources and save it to the data/raw/ folder. No calculations
happen here — just fetching and saving.

Sources used:
  - mfapi.in         : Free Indian MF API (NAV history + scheme metadata)
  - AMFI NAVAll.txt  : Official AMFI file (ISIN codes, categories, live NAV)
  - yfinance         : Yahoo Finance (benchmark index data)

Run this file directly:
  python collect.py

Or call individual functions from run_pipeline.py
"""

import os
import time
import requests
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta


# ─────────────────────────────────────────────────────────────────────────────
# CONFIG — edit these to control what gets collected
# ─────────────────────────────────────────────────────────────────────────────

BASE_DIR   = os.path.dirname(os.path.abspath(__file__))
OUTPUT_DIR = os.path.join(BASE_DIR, "data", "output")   # All raw files land here
NAV_YEARS    = 7                                        # How many years of NAV history to keep
API_DELAY    = 0.5                                      # Seconds to wait between API calls (be polite)
API_BASE_URL = "https://api.mfapi.in/mf"

# Benchmark indices — these are Yahoo Finance tickers
# Used later in calculate.py for Beta, Alpha, and category comparisons
BENCHMARKS = {
    "Nifty 50":           "^NSEI",
    "Nifty 500":          "^CRSLDX",
    "Nifty Midcap 150":   "NIFTY_MID_SELECT.NS",
    "Nifty Smallcap 250": "^CNXSC",
    "Nifty Next 50":      "^NSMIDCP",
    "BSE Sensex":         "^BSESN",
    "Nifty Bank":         "^NSEBANK",
    "India 10Y Bond":     "^IRX",        # Proxy for debt fund benchmark
}

# A focused list of well-known funds to start with.
# scheme_code : AMFI scheme code (get the full list from get_all_schemes())
SAMPLE_FUNDS = {
    # Large Cap
    120503: "Axis Bluechip Fund - Growth",
    100033: "HDFC Top 100 Fund - Growth",
    119598: "Mirae Asset Large Cap Fund - Growth",
    # Flexi Cap
    125354: "Parag Parikh Flexi Cap Fund - Growth",
    120594: "Canara Robeco Flexi Cap Fund - Growth",
    # Mid Cap
    120505: "Axis Midcap Fund - Growth",
    119775: "Kotak Emerging Equity Fund - Growth",
    # Small Cap
    120465: "SBI Small Cap Fund - Growth",
    125497: "Nippon India Small Cap Fund - Growth",
    # Index Funds
    120716: "UTI Nifty 50 Index Fund - Growth",
    120841: "HDFC Index Fund - Nifty 50 Plan - Growth",
    # Hybrid
    119775: "Kotak Equity Hybrid Fund - Growth",
    120838: "ICICI Pru Equity & Debt Fund - Growth",
    # Debt
    119260: "HDFC Short Term Debt Fund - Growth",
    120828: "ICICI Pru Corporate Bond Fund - Growth",
}


# ─────────────────────────────────────────────────────────────────────────────
# UTILITY HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def ensure_output_dir():
    """Create the output directory if it doesn't already exist."""
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"Output folder ready: {OUTPUT_DIR}/")

def safe_get(url, retries=3, delay=2):
    """
    GET request with automatic retry on failure, adding browser headers
    to avoid being blocked by anti-bot systems.
    """
    # 1. Add headers to masquerade as a standard Google Chrome browser
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "application/json, text/plain, */*"
    }

    for attempt in range(retries):
        try:
            # 2. Pass the headers and slightly increase the timeout
            response = requests.get(url, headers=headers, timeout=10)

            # 3. Special handling for 404: Don't retry if the fund just doesn't exist
            if response.status_code == 404:
                print(f"  Attempt {attempt + 1} failed: 404 Not Found (Fund not in database)")
                return None

            response.raise_for_status()
            return response

        except requests.RequestException as e:
            # Attempt to grab the status code if a response object exists
            status_code = response.status_code if 'response' in locals() and response is not None else "Network/Timeout"

            print(f"  Attempt {attempt + 1} failed: {e} (Status: {status_code})")

            if attempt < retries - 1:
                # 4. Progressive Backoff: Wait longer after each failure (2s, then 4s)
                time.sleep(delay * (attempt + 1))

    print(f"  Could not fetch: {url}")
    return None

# def safe_get(url, retries=3, delay=2):
#     """
#     GET request with automatic retry on failure.
#     If the server is temporarily down, it will try again up to 3 times
#     before giving up and returning None.
#     """
#     for attempt in range(retries):
#         try:
#             response = requests.get(url, timeout=7)
#             response.raise_for_status()      # Raises an error for 4xx / 5xx responses
#             return response
#         except requests.RequestException as e:
#             print(f"  Attempt {attempt + 1} failed: {e}")
#             if attempt < retries - 1:
#                 time.sleep(delay)
#     print(f"  Could not fetch: {url}")
#     return None


# ─────────────────────────────────────────────────────────────────────────────
# FUNCTION 1 — Full Scheme List
# Gets every active mutual fund scheme registered with AMFI.
# This is your master lookup table. You pick scheme codes from here.
# ─────────────────────────────────────────────────────────────────────────────

def get_all_schemes():
    """
    Fetches the complete list of all mutual fund schemes from mfapi.in.

    Returns a DataFrame with columns:
        schemeCode | schemeName

    Saved to: data/raw/all_schemes.csv
    """
    print("\n[1] Fetching full scheme list from mfapi.in...")

    response = safe_get(API_BASE_URL)
    if response is None:
        print("  Failed to fetch scheme list.")
        return pd.DataFrame()

    schemes = response.json()
    df = pd.DataFrame(schemes)

    df = df.rename(columns={
        "schemeCode":          "scheme_code",
        "schemeName":          "scheme_name",
        "isinGrowth":          "isin_growth",
        "isinDivReinvestment": "isin_div_reinvestment",
    })

    df["scheme_code"] = df["scheme_code"].astype(int)

    out_path = os.path.join(OUTPUT_DIR, "all_schemes.csv")
    df.to_csv(out_path, index=False)
    print(f"  Saved {len(df):,} schemes → {out_path}")
    return df
# ─────────────────────────────────────────────────────────────────────────────
# FUNCTION 2 — Scheme Metadata
# The /mf/{code} endpoint returns a 'meta' block alongside NAV history.
# That meta block contains the AMC name, category, and scheme type.
# ─────────────────────────────────────────────────────────────────────────────

def get_scheme_metadata(scheme_code):
    """
    Fetches the metadata block for a single scheme from mfapi.in.

    The meta block contains:
        fund_house       : AMC name (e.g. "Axis Mutual Fund")
        scheme_type      : "Open Ended Schemes" / "Close Ended Schemes"
        scheme_category  : Full SEBI category (e.g. "Equity Scheme - Large Cap Fund")
        scheme_code      : Numeric code
        scheme_name      : Full scheme name

    Returns a dict, or None if the request fails.
    """
    url = f"{API_BASE_URL}/{scheme_code}"
    response = safe_get(url)
    if response is None:
        return None

    data = response.json()
    meta = data.get("meta", {})

    # Split the scheme_category into broad_category + sub_category
    # Example: "Equity Scheme - Large Cap Fund"
    #          → broad = "Equity Scheme"  |  sub = "Large Cap Fund"
    full_category = meta.get("scheme_category", "")
    if " - " in full_category:
        parts = full_category.split(" - ", 1)
        broad_category = parts[0].strip()
        sub_category   = parts[1].strip()
    else:
        broad_category = full_category
        sub_category   = ""

    return {
        "scheme_code":      scheme_code,
        "scheme_name":      meta.get("scheme_name", ""),
        "fund_house":       meta.get("fund_house", ""),
        "scheme_type":      meta.get("scheme_type", ""),
        "broad_category":   broad_category,
        "sub_category":     sub_category,
        "full_category":    full_category,
    }


def get_metadata_for_all(fund_dict=SAMPLE_FUNDS):
    """
    Loops through a dictionary of {scheme_code: name} and fetches
    metadata for each. Builds a single metadata table.

    Saved to: data/raw/scheme_metadata.csv
    """
    print("\n[2] Fetching scheme metadata...")

    records = []
    for code, name in fund_dict.items():
        meta = get_scheme_metadata(code)
        if meta:
            records.append(meta)
            print(f"  ✓ {name}")
        else:
            print(f"  ✗ Failed: {code}")
        time.sleep(API_DELAY)

    df = pd.DataFrame(records)
    out_path = os.path.join(OUTPUT_DIR, "scheme_metadata.csv")
    df.to_csv(out_path, index=False)
    print(f"  Saved {len(df)} schemes → {out_path}")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# FUNCTION 3 — NAV History
# The full day-by-day price history for each fund.
# This is the core data that all return metrics are calculated from.
# ─────────────────────────────────────────────────────────────────────────────

def get_nav_history(scheme_code):
    """
    Fetches the complete NAV history for a single scheme from mfapi.in.

    Returns a DataFrame with columns:
        date | nav | scheme_code

    NAVs are sorted oldest → newest.
    Only the last NAV_YEARS years of data are kept.
    """
    url = f"{API_BASE_URL}/{scheme_code}"
    response = safe_get(url)
    if response is None:
        return pd.DataFrame()

    data = response.json()
    nav_records = data.get("data", [])

    if not nav_records:
        return pd.DataFrame()

    df = pd.DataFrame(nav_records)
    df.columns = ["date", "nav"]
    df["date"]        = pd.to_datetime(df["date"], format="%d-%m-%Y")
    df["nav"]         = pd.to_numeric(df["nav"], errors="coerce")
    df["scheme_code"] = scheme_code
    df = df.sort_values("date").reset_index(drop=True)

    # Keep only the last NAV_YEARS years
    cutoff = datetime.today() - timedelta(days=NAV_YEARS * 365)
    df = df[df["date"] >= cutoff]

    return df


def get_nav_history_for_all(fund_dict=SAMPLE_FUNDS):
    """
    Fetches NAV history for every fund in fund_dict.
    Combines them all into one long table (one row per fund per day).

    Saved to: data/raw/nav_history_raw.csv
    """
    print("\n[3] Fetching NAV history...")

    all_nav = []
    for code, name in fund_dict.items():
        print(f"  Fetching: {name}...")
        nav_df = get_nav_history(code)
        if not nav_df.empty:
            nav_df["fund_name"] = name
            all_nav.append(nav_df)
        time.sleep(API_DELAY)

    combined = pd.concat(all_nav, ignore_index=True)
    out_path = os.path.join(OUTPUT_DIR, "nav_history_raw.csv")
    combined.to_csv(out_path, index=False)
    print(f"  Saved {len(combined):,} rows → {out_path}")
    return combined


# ─────────────────────────────────────────────────────────────────────────────
# FUNCTION 4 — Scheme Profile (Derived Features from NAV History)
# Things we can calculate from the NAV series itself —
# inception date, fund age, latest NAV, 52-week high/low.
# These do not require an extra API call — they come from the history.
# ─────────────────────────────────────────────────────────────────────────────

def get_scheme_profile(scheme_code, fund_name=""):
    """
    Derives additional features from a fund's NAV history.

    Features extracted:
        inception_date   : Date of the very first NAV on record
        fund_age_years   : How old the fund is (years since inception)
        latest_nav       : Most recent NAV value
        latest_nav_date  : Date of that NAV
        nav_52w_high     : Highest NAV in the last 52 weeks
        nav_52w_low      : Lowest NAV in the last 52 weeks
        nav_52w_change   : % change from 52-week low to current NAV
        total_nav_days   : Number of trading days on record

    Returns a dict.
    """
    nav_df = get_nav_history(scheme_code)
    if nav_df.empty:
        return None

    nav_df = nav_df.sort_values("date")
    today  = datetime.today()
    cutoff_52w = today - timedelta(weeks=52)
    nav_52w = nav_df[nav_df["date"] >= cutoff_52w]

    latest_nav    = nav_df["nav"].iloc[-1]
    nav_52w_low   = nav_52w["nav"].min()
    nav_52w_high  = nav_52w["nav"].max()
    inception_date = nav_df["date"].iloc[0]
    fund_age_years = (today - inception_date).days / 365.25

    return {
        "scheme_code":      scheme_code,
        "fund_name":        fund_name,
        "inception_date":   inception_date.date(),
        "fund_age_years":   round(fund_age_years, 1),
        "latest_nav":       round(latest_nav, 4),
        "latest_nav_date":  nav_df["date"].iloc[-1].date(),
        "nav_52w_high":     round(nav_52w_high, 4),
        "nav_52w_low":      round(nav_52w_low, 4),
        "nav_52w_change_pct": round((latest_nav - nav_52w_low) / nav_52w_low * 100, 2),
        "total_nav_days":   len(nav_df),
    }


def get_scheme_profiles_for_all(fund_dict=SAMPLE_FUNDS):
    """
    Builds the full scheme profile table for all funds.

    Saved to: data/raw/scheme_profiles.csv
    """
    print("\n[4] Building scheme profiles (NAV-derived features)...")

    profiles = []
    for code, name in fund_dict.items():
        print(f"  Profiling: {name}...")
        profile = get_scheme_profile(code, name)
        if profile:
            profiles.append(profile)
        time.sleep(API_DELAY)

    df = pd.DataFrame(profiles)
    out_path = os.path.join(OUTPUT_DIR, "scheme_profiles.csv")
    df.to_csv(out_path, index=False)
    print(f"  Saved {len(df)} scheme profiles → {out_path}")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# FUNCTION 5 — AMFI Master Data (ISIN codes + Live NAV + Category)
# AMFI publishes a flat text file daily with every fund's current NAV.
# It also contains ISIN codes, which are the universal fund identifiers
# used on stock exchanges and in most third-party platforms.
# ─────────────────────────────────────────────────────────────────────────────

def get_amfi_master_data():
    """
    Downloads and parses the AMFI NAVAll.txt file.
    This is the official end-of-day NAV file published by AMFI every evening.

    Extracts:
        scheme_code      : AMFI numeric code
        isin_growth      : ISIN for the Growth option
        isin_idcw        : ISIN for the IDCW (dividend) option
        scheme_name      : Full scheme name
        nav              : Today's NAV
        nav_date         : Date of that NAV
        broad_category   : Top-level category parsed from section headers
        amc_name         : AMC name parsed from section headers

    Saved to: data/raw/amfi_master.csv
    """
    print("\n[5] Downloading AMFI NAVAll.txt...")

    url = "https://www.amfiindia.com/spages/NAVAll.txt"
    response = safe_get(url)
    if response is None:
        print("  Failed to download AMFI master file.")
        return pd.DataFrame()

    lines = response.text.splitlines()
    records = []
    current_amc      = ""
    current_category = ""

    for line in lines:
        line = line.strip()
        if not line:
            continue

        # Lines with semicolons are fund data rows
        if ";" in line:
            parts = line.split(";")
            if len(parts) < 6:
                continue
            # Check if this looks like a header row (non-numeric scheme code)
            if not parts[0].strip().isdigit():
                continue

            try:
                records.append({
                    "scheme_code":    int(parts[0].strip()),
                    "isin_growth":    parts[1].strip(),
                    "isin_idcw":      parts[2].strip(),
                    "scheme_name":    parts[3].strip(),
                    "nav":            float(parts[4].strip()) if parts[4].strip() not in ["", "N.A."] else None,
                    "nav_date":       parts[5].strip(),
                    "amc_name":       current_amc,
                    "broad_category": current_category,
                })
            except (ValueError, IndexError):
                continue

        # Lines without semicolons are section headers (AMC or category names)
        else:
            # Heuristic: if it contains "Mutual Fund" it is an AMC name
            if "Mutual Fund" in line or "AMC" in line or "Asset Management" in line:
                current_amc = line
            else:
                current_category = line

    df = pd.DataFrame(records)
    if not df.empty:
        df["nav_date"] = pd.to_datetime(df["nav_date"], format="%d-%b-%Y", errors="coerce")

    out_path = os.path.join(OUTPUT_DIR, "amfi_master.csv")
    df.to_csv(out_path, index=False)
    print(f"  Saved {len(df):,} schemes → {out_path}")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# FUNCTION 6 — Benchmark Index Data
# Needed to calculate Beta, Alpha, and to compare fund returns
# against the market index. Fetched from Yahoo Finance via yfinance.
# ─────────────────────────────────────────────────────────────────────────────

def get_benchmark_data(years=NAV_YEARS):
    """
    Downloads historical price data for all benchmark indices
    defined in the BENCHMARKS dictionary at the top of this file.

    Returns a wide DataFrame where each column is one benchmark index.
    Dates are the index.

    Saved to: data/raw/benchmark_data.csv
    """
    print("\n[6] Fetching benchmark index data from Yahoo Finance...")

    start_date = (datetime.today() - timedelta(days=years * 365)).strftime("%Y-%m-%d")
    end_date   = datetime.today().strftime("%Y-%m-%d")

    all_benchmarks = {}

    for name, ticker in BENCHMARKS.items():
        print(f"  Fetching: {name} ({ticker})...")
        try:
            data = yf.download(
                ticker,
                start=start_date,
                end=end_date,
                progress=False,
                auto_adjust=True,      # Adjusts for splits and dividends
            )
            if data.empty:
                print(f"    No data returned for {ticker}")
                continue

            all_benchmarks[name] = data["Close"].squeeze()

        except Exception as e:
            print(f"    Error fetching {ticker}: {e}")
        time.sleep(0.3)

    df = pd.DataFrame(all_benchmarks)
    df.index.name = "date"
    df = df.sort_index()

    out_path = os.path.join(OUTPUT_DIR, "benchmark_data.csv")
    df.to_csv(out_path)
    print(f"  Saved benchmark data ({len(df)} days, {len(df.columns)} indices) → {out_path}")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# FUNCTION 7 — Expense Ratio Data (from AMFI monthly TER disclosure)
# SEBI mandates that every AMC publish Total Expense Ratios monthly.
# AMFI aggregates this into a downloadable file.
# ─────────────────────────────────────────────────────────────────────────────
def get_aum_and_ter(fund_dict=SAMPLE_FUNDS):
    """
    Fetches AUM (₹ crore), expense ratio (%), Morningstar star rating,
    and latest NAV for each fund from mfdata.in.

    This single function replaces the two dead AMFI endpoints:
        ✗ amfiindia.com/modules/AumData   → 404
        ✗ amfiindia.com/modules/TerHtml   → 404
        ✓ mfdata.in/api/v1/schemes/{code} → works, free, no auth

    Saved to: data/raw/aum_and_ter.csv
    """
    print("\n[7] Fetching AUM + TER from mfdata.in...")

    records = []

    for code, name in fund_dict.items():

        # ── Primary: mfdata.in ────────────────────────────────────────────
        url      = f"https://mfdata.in/api/v1/schemes/{code}"
        response = safe_get(url)

        if response is not None:
            try:
                data = response.json().get("data", {})
                records.append({
                    "scheme_code":       code,
                    "fund_name":         name,
                    "aum_cr":            data.get("aum_cr"),
                    "expense_ratio_pct": data.get("expense_ratio"),
                    "morningstar_stars": data.get("morningstar"),
                    "latest_nav":        data.get("nav"),
                    "latest_nav_date":   data.get("nav_date"),
                    "source":            "mfdata.in",
                })
                print(f"  ✓ {name}  [mfdata.in]")
                time.sleep(API_DELAY)
                continue                  # ← skip fallback if primary worked
            except Exception as e:
                print(f"  Parse error from mfdata.in for {name}: {e}")

        # ── Fallback: mfapi.in ────────────────────────────────────────────
        # mfapi.in returns AUM inside the meta block of the scheme detail
        print(f"  → Falling back to mfapi.in for {name}...")
        url      = f"https://api.mfapi.in/mf/{code}"
        response = safe_get(url)

        if response is not None:
            try:
                data = response.json()
                meta = data.get("meta", {})
                records.append({
                    "scheme_code":       code,
                    "fund_name":         name,
                    "aum_cr":            None,   # mfapi.in does not carry AUM
                    "expense_ratio_pct": None,   # mfapi.in does not carry TER
                    "morningstar_stars": None,
                    "latest_nav":        data["data"][0]["nav"] if data.get("data") else None,
                    "latest_nav_date":   data["data"][0]["date"] if data.get("data") else None,
                    "source":            "mfapi.in (fallback — AUM/TER unavailable)",
                })
                print(f"  ✓ {name}  [mfapi.in fallback]")
            except Exception as e:
                print(f"  ✗ Both sources failed for {name}: {e}")
        else:
            print(f"  ✗ Both sources failed for {name}")

        time.sleep(API_DELAY)

    df = pd.DataFrame(records)
    out_path = os.path.join(OUTPUT_DIR, "aum_and_ter.csv")
    df.to_csv(out_path, index=False)
    print(f"\n  Saved {len(df)} records → {out_path}")

    # Warn if AUM data is missing for any fund
    missing = df["aum_cr"].isna().sum()
    if missing > 0:
        print(f"  ⚠ AUM missing for {missing} fund(s) — mfdata.in was unavailable for those.")
    return df

# ─────────────────────────────────────────────────────────────────────────────
# FUNCTION 8 (REVISED) — AUM via AMFI monthly excel file
# Replaces both the dead AMFI AumData and TerHtml endpoints.
# Returns AUM, expense ratio, Morningstar rating, and current NAV
# for each scheme — all from one API call per fund.
# ─────────────────────────────────────────────────────────────────────────────

def get_aum_from_amfi_excel():
    """
    Downloads the AMFI Monthly report Excel file from portal.amfiindia.com.

    Correct URL pattern (confirmed May 2026):
        https://portal.amfiindia.com/spages/am{mon}{year}repo.xls
        e.g. amapr2026repo.xls, ammar2026repo.xls

    Note: AMFI publishes after month end. On May 17 2026, the latest
    available file is April 2026 — May 2026 does not exist yet.

    Saved to: data/raw/aum_amfi_monthly.csv
    """
    print("\n[7-ALT] Fetching AUM from AMFI Monthly Excel...")

    from datetime import datetime, timedelta
    from io import BytesIO

    BASE_URL = "https://portal.amfiindia.com/spages"

    # Month abbreviations exactly as AMFI uses them in filenames
    MONTH_MAP = {
        1: "jan", 2: "feb",  3: "mar", 4: "apr",
        5: "may", 6: "jun",  7: "jul", 8: "aug",
        9: "sep", 10: "oct", 11: "nov", 12: "dec"
    }

    # Try current month first, then walk back up to 3 months
    # (AMFI publishes ~7-10 days after month end so current month
    #  is often unavailable for the first week of a new month)
    today = datetime.today()

    for offset in range(4):
        target     = today.replace(day=1) - timedelta(days=30 * offset)
        mon_str    = MONTH_MAP[target.month]          # e.g. "apr"
        year_str   = target.strftime("%Y")            # e.g. "2026"
        filename   = f"am{mon_str}{year_str}repo.xls"
        url        = f"{BASE_URL}/{filename}"

        print(f"  Trying: {url}")
        response = safe_get(url)

        if response is not None and response.status_code == 200:
            print(f"  ✓ Found: {filename}")

            try:
                df = pd.read_excel(BytesIO(response.content), engine="xlrd")
            except Exception:
                # If xlrd not installed, save raw file and inform user
                raw_path = os.path.join(OUTPUT_DIR, filename)
                with open(raw_path, "wb") as f:
                    f.write(response.content)
                print(f"  Saved raw .xls → {raw_path}")
                print("  Run: pip install xlrd   then re-run this function")
                return pd.DataFrame()

            # Standardise column names
            df.columns = [
                str(c).strip().lower().replace(" ", "_") for c in df.columns
            ]

            print(f"  Columns: {df.columns.tolist()}")
            print(f"  Rows: {len(df)}")

            out_path = os.path.join(OUTPUT_DIR, "aum_amfi_monthly.csv")
            df.to_csv(out_path, index=False)
            print(f"  Saved → {out_path}")
            return df

        print(f"  ✗ Not found — trying previous month...")

    print("  Could not fetch any recent AMFI Monthly file.")
    print("  Download manually from: https://www.amfiindia.com/research-information/amfi-monthly")
    return pd.DataFrame()

# ─────────────────────────────────────────────────────────────────────────────
# FUNCTION 9 — Category Classification Table
# A clean lookup table of SEBI's fund categories.
# Used in Power BI for slicers and category-level comparisons.
# ─────────────────────────────────────────────────────────────────────────────

def build_category_reference():
    """
    Builds a static reference table of SEBI's official mutual fund categories.
    These are SEBI-mandated as of October 2017 (Circular SEBI/HO/IMD/DF3/CIR/P/2017/114).

    This table is used in Power BI to:
        - Drive the Category slicer
        - Group funds for peer comparison
        - Assign benchmark indices per category

    Saved to: data/raw/category_reference.csv
    """
    print("\n[9] Building SEBI category reference table...")

    categories = [
        # Broad | Sub-category | Typical benchmark | Risk level
        ("Equity Scheme",       "Large Cap Fund",               "Nifty 100",              "Moderately High"),
        ("Equity Scheme",       "Large & Mid Cap Fund",         "Nifty LargeMidcap 250",  "Moderately High"),
        ("Equity Scheme",       "Mid Cap Fund",                 "Nifty Midcap 150",       "High"),
        ("Equity Scheme",       "Small Cap Fund",               "Nifty Smallcap 250",     "Very High"),
        ("Equity Scheme",       "Multi Cap Fund",               "Nifty 500",              "High"),
        ("Equity Scheme",       "Flexi Cap Fund",               "Nifty 500",              "Moderately High"),
        ("Equity Scheme",       "ELSS",                         "Nifty 500",              "High"),
        ("Equity Scheme",       "Sectoral / Thematic Fund",     "Nifty 500",              "Very High"),
        ("Equity Scheme",       "Focused Fund",                 "Nifty 500",              "High"),
        ("Equity Scheme",       "Dividend Yield Fund",          "Nifty Dividend Opp 50",  "Moderately High"),
        ("Equity Scheme",       "Value Fund",                   "Nifty 500 Value 50",     "Moderately High"),
        ("Equity Scheme",       "Contra Fund",                  "Nifty 500",              "Moderately High"),
        ("Hybrid Scheme",       "Aggressive Hybrid Fund",       "Nifty 50 Hybrid 65:35",  "Moderately High"),
        ("Hybrid Scheme",       "Conservative Hybrid Fund",     "Nifty 50 Hybrid 25:75",  "Moderate"),
        ("Hybrid Scheme",       "Balanced Hybrid Fund",         "Nifty 50 Hybrid 50:50",  "Moderate"),
        ("Hybrid Scheme",       "Dynamic Asset Allocation",     "Nifty 50",               "Moderate"),
        ("Hybrid Scheme",       "Multi Asset Allocation",       "Nifty 50",               "Moderate"),
        ("Hybrid Scheme",       "Arbitrage Fund",               "Nifty 50 Arbitrage",     "Low"),
        ("Hybrid Scheme",       "Equity Savings Fund",          "Nifty Equity Savings",   "Low to Moderate"),
        ("Debt Scheme",         "Overnight Fund",               "CRISIL Overnight",       "Low"),
        ("Debt Scheme",         "Liquid Fund",                  "CRISIL Liquid",          "Low"),
        ("Debt Scheme",         "Ultra Short Duration Fund",    "CRISIL Ultra Short",     "Low to Moderate"),
        ("Debt Scheme",         "Low Duration Fund",            "CRISIL Low Duration",    "Low to Moderate"),
        ("Debt Scheme",         "Money Market Fund",            "CRISIL Money Market",    "Low to Moderate"),
        ("Debt Scheme",         "Short Duration Fund",          "CRISIL Short Term",      "Moderate"),
        ("Debt Scheme",         "Medium Duration Fund",         "CRISIL Medium Term",     "Moderate"),
        ("Debt Scheme",         "Medium to Long Duration Fund", "CRISIL Composite Bond",  "Moderate"),
        ("Debt Scheme",         "Long Duration Fund",           "CRISIL Long Term Gilt",  "Moderately High"),
        ("Debt Scheme",         "Dynamic Bond Fund",            "CRISIL Composite Bond",  "Moderate"),
        ("Debt Scheme",         "Corporate Bond Fund",          "CRISIL Corporate Bond",  "Moderate"),
        ("Debt Scheme",         "Credit Risk Fund",             "CRISIL Credit Risk",     "High"),
        ("Debt Scheme",         "Banking & PSU Fund",           "CRISIL Banking & PSU",   "Moderate"),
        ("Debt Scheme",         "Gilt Fund",                    "CRISIL Gilt",            "Moderate"),
        ("Debt Scheme",         "Floater Fund",                 "CRISIL Short Term",      "Low to Moderate"),
        ("Index / ETF Scheme",  "Index Fund - Nifty 50",        "Nifty 50",               "Moderately High"),
        ("Index / ETF Scheme",  "Index Fund - Nifty Next 50",   "Nifty Next 50",          "High"),
        ("Index / ETF Scheme",  "Index Fund - Nifty Midcap",    "Nifty Midcap 150",       "High"),
        ("Index / ETF Scheme",  "Index Fund - Sensex",          "BSE Sensex",             "Moderately High"),
        ("Index / ETF Scheme",  "Gold ETF",                     "Domestic Gold Price",    "Moderate"),
        ("Solution Oriented",   "Retirement Fund",              "Nifty 50",               "Varies"),
        ("Solution Oriented",   "Children's Fund",              "Nifty 50",               "Varies"),
    ]

    df = pd.DataFrame(categories, columns=[
        "broad_category", "sub_category", "benchmark_index", "risk_level"
    ])

    out_path = os.path.join(OUTPUT_DIR, "category_reference.csv")
    df.to_csv(out_path, index=False)
    print(f"  Saved {len(df)} categories → {out_path}")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# FUNCTION 10 — Rolling NAV Snapshot (for momentum analysis)
# Captures NAV values at fixed points in the past.
# Used in Power BI to show 1M / 3M / 6M / 1Y point-in-time returns
# without recalculating the full series every time.
# ─────────────────────────────────────────────────────────────────────────────

def get_nav_snapshots(fund_dict=SAMPLE_FUNDS):
    """
    For each fund, captures the NAV at fixed lookback points:
        today, 1 month ago, 3 months ago, 6 months ago, 1 year ago, 3 years ago

    These snapshots are used in Power BI to calculate period returns quickly
    without having to process the full NAV series in DAX.

    Saved to: data/raw/nav_snapshots.csv
    """
    print("\n[10] Capturing NAV snapshots at fixed dates...")

    today = datetime.today()
    lookbacks = {
        "nav_today":   today,
        "nav_1m_ago":  today - timedelta(days=30),
        "nav_3m_ago":  today - timedelta(days=91),
        "nav_6m_ago":  today - timedelta(days=182),
        "nav_1y_ago":  today - timedelta(days=365),
        "nav_3y_ago":  today - timedelta(days=1095),
        "nav_5y_ago":  today - timedelta(days=1825),
    }

    records = []

    for code, name in fund_dict.items():
        nav_df = get_nav_history(code)
        if nav_df.empty:
            continue

        nav_df = nav_df.sort_values("date").set_index("date")
        record = {"scheme_code": code, "fund_name": name}

        for label, target_date in lookbacks.items():
            # Get the closest available NAV on or before the target date
            past = nav_df[nav_df.index <= pd.Timestamp(target_date)]
            record[label] = round(past["nav"].iloc[-1], 4) if not past.empty else None

        records.append(record)
        print(f"  ✓ {name}")
        time.sleep(API_DELAY)

    df = pd.DataFrame(records)
    out_path = os.path.join(OUTPUT_DIR, "nav_snapshots.csv")
    df.to_csv(out_path, index=False)
    print(f"  Saved {len(df)} snapshot records → {out_path}")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# MASTER RUNNER — calls all functions in the correct order
# ─────────────────────────────────────────────────────────────────────────────

def run_all():
    """
    Runs the full data collection pipeline in the correct sequence.
    Each function saves its own output file to data/raw/.
    """
    print("=" * 60)
    print("  MUTUAL FUND DASHBOARD — DATA COLLECTION")
    print(f"  Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 60)

    ensure_output_dir()

    get_all_schemes()              # → all_schemes.csv
    get_metadata_for_all()         # → scheme_metadata.csv
    get_nav_history_for_all()      # → nav_history_raw.csv
    get_scheme_profiles_for_all()  # → scheme_profiles.csv
    get_amfi_master_data()         # → amfi_master.csv
    get_benchmark_data()           # → benchmark_data.csv
    #get_aum_and_ter()              # → aum_and_ter.csv  (replaces dead AMFI AumData + TerHtml)
    get_aum_from_amfi_excel()      # -> aum_amfi_monthly.csv
    build_category_reference()     # → category_reference.csv
    get_nav_snapshots()            # → nav_snapshots.csv

    print("\n" + "=" * 60)
    print("  COLLECTION COMPLETE")
    print(f"  Finished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"  All files saved to: {OUTPUT_DIR}/")
    print("=" * 60)


if __name__ == "__main__":
    run_all()

  MUTUAL FUND DASHBOARD — DATA COLLECTION
  Started: 2026-05-20 01:19:10
Output folder ready: data/raw/

[1] Fetching full scheme list from mfapi.in...
  Saved 37,601 schemes → data/raw\all_schemes.csv

[2] Fetching scheme metadata...
  ✓ Axis Bluechip Fund - Growth
  ✓ HDFC Top 100 Fund - Growth
  ✓ Mirae Asset Large Cap Fund - Growth
  ✓ Parag Parikh Flexi Cap Fund - Growth
  ✓ Canara Robeco Flexi Cap Fund - Growth
  ✓ Axis Midcap Fund - Growth
  ✓ Kotak Equity Hybrid Fund - Growth
  ✓ SBI Small Cap Fund - Growth
  ✓ Nippon India Small Cap Fund - Growth
  ✓ UTI Nifty 50 Index Fund - Growth
  ✓ HDFC Index Fund - Nifty 50 Plan - Growth
  ✓ ICICI Pru Equity & Debt Fund - Growth
  ✓ HDFC Short Term Debt Fund - Growth
  ✓ ICICI Pru Corporate Bond Fund - Growth
  Saved 14 schemes → data/raw\scheme_metadata.csv

[3] Fetching NAV history...
  Fetching: Axis Bluechip Fund - Growth...
  Fetching: HDFC Top 100 Fund - Growth...
  Fetching: Mirae Asset Large Cap Fund - Growth...
  Fetching: Parag

In [8]:
"""
calculate.py — Metrics Calculation Engine
==========================================
Mutual Fund Analysis Dashboard — Stage 3
------------------------------------------
Reads all clean CSV files and produces two output files:

    data/output/fund_metrics.csv     ← Main fact table for Power BI (one row per fund)
    data/output/rolling_returns.csv  ← Rolling 1Y returns over time (for line charts)

Every number that appears in the Power BI dashboard is born here.
No data is fetched from the internet in this file — it only reads
from data/clean/ and writes to data/output/.

Run this file directly:
    python calculate.py

Or it will be called automatically by run_pipeline.py
"""

import os
import numpy as np
import pandas as pd

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────

#BASE_DIR   = os.path.dirname(os.path.abspath(__file__))  Uncomment this line of code when converting block to its own separate file
BASE_DIR   = os.getcwd()
CLEAN_DIR  = os.path.join(BASE_DIR, "data", "cleaned")
OUTPUT_DIR = os.path.join(BASE_DIR, "data", "output")

# RBI 91-day T-Bill rate — update this periodically
# Current rate as of May 2026: ~7.1%
RISK_FREE_RATE_ANNUAL = 0.071

# Number of trading days in a year (used to annualise daily figures)
TRADING_DAYS = 252

# Minimum number of NAV data points needed to calculate a metric
# 252 = 1 year, 756 = 3 years, 1260 = 5 years
MIN_DAYS_1Y = 240
MIN_DAYS_3Y = 700
MIN_DAYS_5Y = 1200

# Maps each fund sub_category to the most appropriate benchmark
# from benchmark_data_clean.csv
CATEGORY_BENCHMARK_MAP = {
    # Equity
    "Large Cap Fund":                       "Nifty 50",
    "Large & Mid Cap Fund":                 "Nifty 500",
    "Mid Cap Fund":                         "Nifty Midcap 150",
    "Small Cap Fund":                       "Nifty Smallcap 250",
    "Multi Cap Fund":                       "Nifty 500",
    "Flexi Cap Fund":                       "Nifty 500",
    "ELSS":                                 "Nifty 500",
    "Focused Fund":                         "Nifty 500",
    "Dividend Yield Fund":                  "Nifty 50",
    "Value Fund":                           "Nifty 500",
    "Value Fund/Contra Fund":               "Nifty 500",
    "Contra Fund":                          "Nifty 500",
    "Sectoral / Thematic Fund":             "Nifty 500",
    "Sectoral/Thematic Funds":              "Nifty 500",
    # Hybrid
    "Aggressive Hybrid Fund":               "Nifty 50",
    "Balanced Hybrid Fund":                 "Nifty 50",
    "Conservative Hybrid Fund":             "India 10Y Bond",
    "Dynamic Asset Allocation":             "Nifty 50",
    "Balanced Advantage Fund":              "Nifty 50",
    "Multi Asset Allocation":               "Nifty 50",
    "Multi Asset Allocation Fund":          "Nifty 50",
    "Arbitrage Fund":                       "Nifty 50",
    "Equity Savings Fund":                  "Nifty 50",
    # Index / ETF
    "Index Fund - Nifty 50":               "Nifty 50",
    "Index Fund - Nifty Next 50":          "Nifty Next 50",
    "Index Fund - Nifty Midcap":           "Nifty Midcap 150",
    "Index Fund - Sensex":                 "BSE Sensex",
    "Index Funds":                         "Nifty 50",
    # Debt
    "Overnight Fund":                       "India 10Y Bond",
    "Liquid Fund":                          "India 10Y Bond",
    "Ultra Short Duration Fund":            "India 10Y Bond",
    "Low Duration Fund":                    "India 10Y Bond",
    "Money Market Fund":                    "India 10Y Bond",
    "Short Duration Fund":                  "India 10Y Bond",
    "Medium Duration Fund":                 "India 10Y Bond",
    "Medium to Long Duration Fund":         "India 10Y Bond",
    "Long Duration Fund":                   "India 10Y Bond",
    "Dynamic Bond Fund":                    "India 10Y Bond",
    "Corporate Bond Fund":                  "India 10Y Bond",
    "Credit Risk Fund":                     "India 10Y Bond",
    "Banking & PSU Fund":                   "India 10Y Bond",
    "Banking and PSU Fund":                 "India 10Y Bond",
    "Gilt Fund":                            "India 10Y Bond",
    "Gilt Fund with 10 year constant duration": "India 10Y Bond",
    "Floater Fund":                         "India 10Y Bond",
    # Solution Oriented
    "Retirement Fund":                      "Nifty 50",
    "Children's Fund":                      "Nifty 50",
    "Childrens Fund":                       "Nifty 50",
}

DEFAULT_BENCHMARK = "Nifty 50"


# ─────────────────────────────────────────────────────────────────────────────
# UTILITY HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def ensure_output_dir():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"Output folder ready: {OUTPUT_DIR}/")


def load_clean_files():
    """
    Loads all clean CSV files into DataFrames.
    Returns a dictionary keyed by a short name for each file.
    """
    print("\n[Loading clean files...]")

    files = {
        "nav_history":       "nav_history_clean.csv",
        "nav_snapshots":     "nav_snapshots_clean.csv",
        "scheme_metadata":   "scheme_metadata_clean.csv",
        "scheme_profiles":   "scheme_profiles_clean.csv",
        "benchmarks":        "benchmark_data_clean.csv",
        "category_ref":      "category_reference_clean.csv",
        "aum_monthly":       "aum_amfi_monthly_clean.csv",
    }

    loaded = {}
    for key, filename in files.items():
        path = os.path.join(CLEAN_DIR, filename)
        if not os.path.exists(path):
            print(f"  ✗ Not found: {filename} — skipping")
            loaded[key] = pd.DataFrame()
            continue
        df = pd.read_csv(path)
        # Parse date columns automatically
        for col in df.columns:
            if "date" in col.lower():
                df[col] = pd.to_datetime(df[col], errors="coerce")
        loaded[key] = df
        print(f"  ✓ {filename:<40} {df.shape}")

    return loaded


# ─────────────────────────────────────────────────────────────────────────────
# CORE METRICS — calculated from the full NAV series
# ─────────────────────────────────────────────────────────────────────────────

def calc_cagr(nav_series, years):
    """
    Compounded Annual Growth Rate over a given number of years.

    Formula: (End NAV / Start NAV) ^ (1 / years) - 1

    Args:
        nav_series : pandas Series of NAV values, sorted oldest first
        years      : lookback period in years (1, 3, or 5)

    Returns:
        CAGR as a percentage (e.g. 14.5 means 14.5%), or None if
        insufficient history exists.
    """
    days_needed = int(years * TRADING_DAYS)
    if len(nav_series) < days_needed:
        return None

    end_nav   = nav_series.iloc[-1]
    start_nav = nav_series.iloc[-days_needed]

    if start_nav <= 0:
        return None

    cagr = ((end_nav / start_nav) ** (1 / years) - 1) * 100
    return round(cagr, 4)


def calc_volatility(daily_returns):
    """
    Annualised standard deviation of daily returns.
    Measures the total risk (both upside and downside swings).

    Returns: Volatility as a percentage.
    """
    if len(daily_returns) < MIN_DAYS_1Y:
        return None
    vol = daily_returns.std() * np.sqrt(TRADING_DAYS) * 100
    return round(vol, 4)


def calc_sharpe_ratio(daily_returns):
    """
    Sharpe Ratio: (Annualised Return - Risk-Free Rate) / Annualised Volatility

    Measures return earned per unit of TOTAL risk.
    Above 1.0 is considered good. Above 2.0 is excellent.

    Returns: Sharpe Ratio (dimensionless), or None.
    """
    if len(daily_returns) < MIN_DAYS_1Y:
        return None

    annual_return = daily_returns.mean() * TRADING_DAYS
    annual_vol    = daily_returns.std()  * np.sqrt(TRADING_DAYS)

    if annual_vol == 0:
        return None

    sharpe = (annual_return - RISK_FREE_RATE_ANNUAL) / annual_vol
    return round(sharpe, 4)


def calc_sortino_ratio(daily_returns):
    """
    Sortino Ratio: (Annualised Return - Risk-Free Rate) / Downside Deviation

    Like Sharpe but only penalises DOWNSIDE volatility.
    More appropriate for equity funds where upside swings are desirable.

    Returns: Sortino Ratio (dimensionless), or None.
    """
    if len(daily_returns) < MIN_DAYS_1Y:
        return None

    annual_return   = daily_returns.mean() * TRADING_DAYS
    downside_returns = daily_returns[daily_returns < 0]

    if len(downside_returns) == 0:
        return None

    downside_dev = downside_returns.std() * np.sqrt(TRADING_DAYS)

    if downside_dev == 0:
        return None

    sortino = (annual_return - RISK_FREE_RATE_ANNUAL) / downside_dev
    return round(sortino, 4)


def calc_max_drawdown(nav_series):
    """
    Maximum Drawdown: the worst peak-to-trough decline in NAV.

    Formula: min((NAV - Rolling Peak) / Rolling Peak)

    Example: A fund whose NAV fell from 100 to 60 has a max drawdown of -40%.
    This is always a negative number. Closer to 0 is better.

    Returns: Max drawdown as a percentage (e.g. -32.5 means -32.5%).
    """
    if len(nav_series) < 10:
        return None

    rolling_peak = nav_series.cummax()
    drawdown     = (nav_series - rolling_peak) / rolling_peak
    return round(drawdown.min() * 100, 4)


def calc_beta_alpha(fund_daily_returns, benchmark_daily_returns):
    """
    Beta: how much the fund moves relative to its benchmark.
        Beta = 1.0  → moves exactly with the market
        Beta > 1.0  → more volatile than the market (aggressive)
        Beta < 1.0  → less volatile than the market (defensive)

    Alpha: excess return above what Beta alone would predict.
        Positive alpha → fund manager is adding value
        Negative alpha → fund manager is destroying value

    Returns: (beta, alpha_pct) tuple, or (None, None).
    """
    # Align both series on the same dates
    aligned = pd.concat([fund_daily_returns, benchmark_daily_returns], axis=1, sort=False).dropna()
    aligned.columns = ["fund", "bench"]

    if len(aligned) < MIN_DAYS_1Y:
        return None, None

    cov_matrix = np.cov(aligned["fund"], aligned["bench"])
    bench_var  = cov_matrix[1][1]

    if bench_var == 0:
        return None, None

    beta  = cov_matrix[0][1] / bench_var
    # Annualise alpha: Jensen's Alpha = Rp - [Rf + Beta * (Rm - Rf)]
    rp    = aligned["fund"].mean()  * TRADING_DAYS          # Fund annual return
    rm    = aligned["bench"].mean() * TRADING_DAYS          # Benchmark annual return
    alpha = (rp - (RISK_FREE_RATE_ANNUAL + beta * (rm - RISK_FREE_RATE_ANNUAL))) * 100

    return round(beta, 4), round(alpha, 4)


def calc_tracking_error(fund_daily_returns, benchmark_daily_returns):
    """
    Tracking Error: standard deviation of (fund return - benchmark return).

    Primarily used for index funds to measure how closely they replicate
    their benchmark. Lower is better for passive funds.

    Returns: Annualised tracking error as a percentage.
    """
    aligned = pd.concat([fund_daily_returns, benchmark_daily_returns], axis=1, sort=False).dropna()
    aligned.columns = ["fund", "bench"]

    if len(aligned) < MIN_DAYS_1Y:
        return None

    diff = aligned["fund"] - aligned["bench"]
    te   = diff.std() * np.sqrt(TRADING_DAYS) * 100
    return round(te, 4)


def calc_calmar_ratio(cagr_3y, max_drawdown):
    """
    Calmar Ratio: CAGR / |Max Drawdown|

    Measures how much return the fund generates per unit of drawdown risk.
    Higher is better. Above 1.0 is generally good.

    Returns: Calmar ratio (dimensionless), or None.
    """
    if cagr_3y is None or max_drawdown is None or max_drawdown == 0:
        return None
    calmar = cagr_3y / abs(max_drawdown)
    return round(calmar, 4)


# ─────────────────────────────────────────────────────────────────────────────
# SNAPSHOT RETURNS — fast period returns from nav_snapshots_clean.csv
# These complement the full NAV series calculations above.
# ─────────────────────────────────────────────────────────────────────────────

def calc_snapshot_returns(snapshots_df):
    """
    Calculates period returns from the pre-built NAV snapshot table.
    Much faster than recalculating from full NAV history.

    Snapshot columns used:
        nav_today, nav_1m_ago, nav_3m_ago, nav_6m_ago,
        nav_1y_ago, nav_3y_ago, nav_5y_ago

    Returns a DataFrame with one row per fund and these new columns:
        return_1m_pct    : 1-month absolute return
        return_3m_pct    : 3-month absolute return
        return_6m_pct    : 6-month absolute return
        return_1y_pct    : 1-year absolute return
        cagr_3y_snap     : 3-year CAGR from snapshots (cross-check)
        cagr_5y_snap     : 5-year CAGR from snapshots (cross-check)
    """
    df = snapshots_df.copy()

    def pct_change(today, past):
        if pd.isna(today) or pd.isna(past) or past == 0:
            return None
        return round((today - past) / past * 100, 4)

    def cagr_snap(today, past, years):
        if pd.isna(today) or pd.isna(past) or past == 0:
            return None
        return round(((today / past) ** (1 / years) - 1) * 100, 4)

    df["return_1m_pct"]  = df.apply(lambda r: pct_change(r["nav_today"], r["nav_1m_ago"]),  axis=1)
    df["return_3m_pct"]  = df.apply(lambda r: pct_change(r["nav_today"], r["nav_3m_ago"]),  axis=1)
    df["return_6m_pct"]  = df.apply(lambda r: pct_change(r["nav_today"], r["nav_6m_ago"]),  axis=1)
    df["return_1y_pct"]  = df.apply(lambda r: pct_change(r["nav_today"], r["nav_1y_ago"]),  axis=1)
    df["cagr_3y_snap"]   = df.apply(lambda r: cagr_snap(r["nav_today"], r["nav_3y_ago"], 3), axis=1)
    df["cagr_5y_snap"]   = df.apply(lambda r: cagr_snap(r["nav_today"], r["nav_5y_ago"], 5), axis=1)

    return df[["scheme_code", "return_1m_pct", "return_3m_pct",
               "return_6m_pct", "return_1y_pct", "cagr_3y_snap", "cagr_5y_snap"]]


# ─────────────────────────────────────────────────────────────────────────────
# COMPOSITE SCORE — ranks funds within their category
# ─────────────────────────────────────────────────────────────────────────────

def calc_composite_score(df):
    """
    Builds a single 0–100 composite score for each fund.
    Used to produce the "Top Performers" view in Power BI.

    Weights (must sum to 1.0):
        35% Sharpe Ratio      — risk-adjusted return quality
        30% CAGR 3Y           — medium-term return
        20% Max Drawdown      — downside protection (inverted: lower loss = higher score)
        15% Volatility        — stability (inverted: lower = higher score)

    Scoring method: Min-Max normalisation per metric, scaled 0–100.
    All normalisations are done WITHIN the full fund universe so scores
    are relative to peers, not absolute.
    """

    def minmax(series, invert=False):
        """Normalise a series to 0–100. Invert=True when lower raw value = better score."""
        mn, mx = series.min(), series.max()
        if mx == mn:
            return pd.Series([50.0] * len(series), index=series.index)
        norm = (series - mn) / (mx - mn) * 100
        return (100 - norm) if invert else norm

    metrics = df[["scheme_code", "sharpe_ratio", "cagr_3y", "max_drawdown_pct", "volatility_pct"]].copy()
    metrics = metrics.dropna(subset=["sharpe_ratio", "cagr_3y", "max_drawdown_pct", "volatility_pct"])

    metrics["sharpe_norm"]    = minmax(metrics["sharpe_ratio"],    invert=False)
    metrics["cagr_norm"]      = minmax(metrics["cagr_3y"],         invert=False)
    metrics["drawdown_norm"]  = minmax(metrics["max_drawdown_pct"],invert=True)  # less negative = better
    metrics["vol_norm"]       = minmax(metrics["volatility_pct"],  invert=True)  # lower = better

    metrics["composite_score"] = (
        0.35 * metrics["sharpe_norm"]   +
        0.30 * metrics["cagr_norm"]     +
        0.20 * metrics["drawdown_norm"] +
        0.15 * metrics["vol_norm"]
    ).round(2)

    # Rank within entire universe (1 = best)
    metrics["universe_rank"] = metrics["composite_score"].rank(ascending=False, method="min").astype(int)

    return df.merge(
        metrics[["scheme_code", "composite_score", "universe_rank"]],
        on="scheme_code", how="left"
    )


# ─────────────────────────────────────────────────────────────────────────────
# ROLLING RETURNS — for the NAV trend line chart in Power BI
# ─────────────────────────────────────────────────────────────────────────────

def calc_rolling_returns(nav_history_df, window=252):
    """
    Calculates rolling 1-year returns for every fund over its full NAV history.
    This produces a time-series table used in Power BI line charts to show
    how consistent each fund's returns have been over time.

    Args:
        nav_history_df : the full nav_history_clean DataFrame
        window         : rolling window in trading days (252 = 1 year)

    Returns a long-format DataFrame:
        date | scheme_code | fund_name | rolling_1y_return_pct

    Saved to: data/output/rolling_returns.csv
    """
    print("\n  Calculating rolling 1Y returns...")

    results = []

    for (code, name), group in nav_history_df.groupby(["scheme_code", "fund_name"]):
        group = group.sort_values("date").copy()
        group["rolling_1y_return_pct"] = (
            group["nav"].pct_change(periods=window) * 100
        ).round(4)
        group["scheme_code"] = code
        group["fund_name"]   = name
        results.append(group[["date", "scheme_code", "fund_name", "rolling_1y_return_pct"]])

    combined = pd.concat(results, ignore_index=True).dropna(subset=["rolling_1y_return_pct"])

    out_path = os.path.join(OUTPUT_DIR, "rolling_returns.csv")
    combined.to_csv(out_path, index=False)
    print(f"  Saved {len(combined):,} rows → {out_path}")
    return combined


# ─────────────────────────────────────────────────────────────────────────────
# MAIN CALCULATION RUNNER
# ─────────────────────────────────────────────────────────────────────────────

def run_all_calculations():
    """
    Master function. Runs every metric calculation for every fund and
    assembles the final fund_metrics.csv fact table.

    Output columns (fund_metrics.csv):
    ─────────────────────────────────
    Identity:
        scheme_code, fund_name, fund_house, broad_category, sub_category
        scheme_type, full_category, benchmark_used

    Return metrics:
        cagr_1y, cagr_3y, cagr_5y        (from full NAV series)
        cagr_3y_snap, cagr_5y_snap       (cross-check from snapshots)
        return_1m_pct, return_3m_pct,
        return_6m_pct, return_1y_pct     (from snapshots)

    Risk metrics:
        volatility_pct                   (annualised std dev of daily returns)
        max_drawdown_pct                 (worst peak-to-trough decline)
        beta                             (vs category benchmark)
        alpha_pct                        (annualised Jensen's alpha)
        tracking_error_pct              (vs category benchmark)

    Risk-adjusted return metrics:
        sharpe_ratio
        sortino_ratio
        calmar_ratio

    Composite score:
        composite_score                  (0–100, weighted blend)
        universe_rank                    (1 = best in full universe)

    Profile fields (merged from scheme_profiles):
        inception_date, fund_age_years
        latest_nav, nav_52w_high, nav_52w_low, nav_52w_change_pct
        total_nav_days
    """

    print("=" * 60)
    print("  MUTUAL FUND DASHBOARD — METRICS CALCULATION")
    print("=" * 60)

    ensure_output_dir()
    data = load_clean_files()

    nav_df       = data["nav_history"]
    snap_df      = data["nav_snapshots"]
    meta_df      = data["scheme_metadata"]
    profiles_df  = data["scheme_profiles"]
    bench_df     = data["benchmarks"]

    # Prepare benchmark data — index by date for fast lookup
    bench_df = bench_df.set_index("date")

    # Get list of funds to process
    funds = nav_df[["scheme_code", "fund_name"]].drop_duplicates()
    print(f"\n  Funds to process: {len(funds)}")

    # ── Step 1: Calculate per-fund metrics from full NAV series ──────────
    print("\n[Step 1] Calculating core metrics from NAV history...")

    all_metrics = []

    for _, row in funds.iterrows():
        code = row["scheme_code"]
        name = row["fund_name"]

        # Get this fund's NAV history, sorted oldest → newest
        fund_nav = (
            nav_df[nav_df["scheme_code"] == code]
            .sort_values("date")
            .set_index("date")["nav"]
        )

        # Daily returns (percentage change day-over-day)
        daily_returns = fund_nav.pct_change().dropna()

        # Look up which sub_category this fund belongs to
        meta_row   = meta_df[meta_df["scheme_code"] == code]
        sub_cat    = meta_row["sub_category"].iloc[0] if not meta_row.empty else ""
        bench_name = CATEGORY_BENCHMARK_MAP.get(sub_cat, DEFAULT_BENCHMARK)

        # Get benchmark daily returns aligned to this fund's date range
        bench_series = bench_df[bench_name].dropna() if bench_name in bench_df.columns else pd.Series()
        bench_returns = bench_series.pct_change().dropna()

        # ── Calculate all metrics ─────────────────────────────────────────
        cagr_1y = calc_cagr(fund_nav, 1)
        cagr_3y = calc_cagr(fund_nav, 3)
        cagr_5y = calc_cagr(fund_nav, 5)
        vol     = calc_volatility(daily_returns)
        sharpe  = calc_sharpe_ratio(daily_returns)
        sortino = calc_sortino_ratio(daily_returns)
        mdd     = calc_max_drawdown(fund_nav)
        calmar  = calc_calmar_ratio(cagr_3y, mdd)
        beta, alpha = calc_beta_alpha(daily_returns, bench_returns)
        te      = calc_tracking_error(daily_returns, bench_returns)

        all_metrics.append({
            "scheme_code":          code,
            "fund_name":            name,
            "benchmark_used":       bench_name,
            "cagr_1y":              cagr_1y,
            "cagr_3y":              cagr_3y,
            "cagr_5y":              cagr_5y,
            "volatility_pct":       vol,
            "sharpe_ratio":         sharpe,
            "sortino_ratio":        sortino,
            "max_drawdown_pct":     mdd,
            "calmar_ratio":         calmar,
            "beta":                 beta,
            "alpha_pct":            alpha,
            "tracking_error_pct":   te,
        })

        print(f"  ✓ {name:<50} Sharpe: {sharpe}  CAGR 3Y: {cagr_3y}%")

    metrics_df = pd.DataFrame(all_metrics)

    # ── Step 2: Add snapshot-based returns ───────────────────────────────
    print("\n[Step 2] Calculating snapshot period returns...")
    snap_returns = calc_snapshot_returns(snap_df)
    metrics_df   = metrics_df.merge(snap_returns, on="scheme_code", how="left")

    # ── Step 3: Merge scheme metadata ────────────────────────────────────
    print("\n[Step 3] Merging scheme metadata...")
    meta_cols = ["scheme_code", "fund_house", "broad_category",
                 "sub_category", "scheme_type", "full_category"]
    meta_slim = meta_df[[c for c in meta_cols if c in meta_df.columns]]
    metrics_df = metrics_df.merge(meta_slim, on="scheme_code", how="left")

    # ── Step 4: Merge scheme profiles ────────────────────────────────────
    print("\n[Step 4] Merging scheme profiles...")
    profile_cols = ["scheme_code", "inception_date", "fund_age_years",
                    "latest_nav", "nav_52w_high", "nav_52w_low",
                    "nav_52w_change_pct", "total_nav_days"]
    profile_slim = profiles_df[[c for c in profile_cols if c in profiles_df.columns]]
    metrics_df   = metrics_df.merge(profile_slim, on="scheme_code", how="left")

    # ── Step 5: Add composite score and universe rank ─────────────────────
    print("\n[Step 5] Calculating composite scores and universe rank...")
    metrics_df = calc_composite_score(metrics_df)

    # ── Step 6: Add category rank (rank within sub_category peers) ────────
    print("\n[Step 6] Calculating within-category peer rank...")
    metrics_df["category_rank"] = (
        metrics_df.groupby("sub_category")["composite_score"]
        .rank(ascending=False, method="min")
        .astype("Int64")      # Int64 supports NaN, int does not
    )

    # ── Step 7: Rolling returns (separate file for line charts) ──────────
    print("\n[Step 7] Generating rolling returns table...")
    calc_rolling_returns(nav_df)

    # ── Step 8: Column ordering and final save ───────────────────────────
    print("\n[Step 8] Saving fund_metrics.csv...")

    # Define clean column order for Power BI
    col_order = [
        # Identity
        "scheme_code", "fund_name", "fund_house",
        "broad_category", "sub_category", "full_category",
        "scheme_type", "benchmark_used",
        # Profile
        "inception_date", "fund_age_years", "total_nav_days",
        "latest_nav", "nav_52w_high", "nav_52w_low", "nav_52w_change_pct",
        # Returns (from NAV series)
        "cagr_1y", "cagr_3y", "cagr_5y",
        # Returns (from snapshots)
        "return_1m_pct", "return_3m_pct", "return_6m_pct",
        "return_1y_pct", "cagr_3y_snap", "cagr_5y_snap",
        # Risk
        "volatility_pct", "max_drawdown_pct", "beta", "tracking_error_pct",
        # Risk-adjusted
        "sharpe_ratio", "sortino_ratio", "calmar_ratio", "alpha_pct",
        # Composite
        "composite_score", "universe_rank", "category_rank",
    ]

    # Keep only columns that exist (guards against optional fields being absent)
    col_order  = [c for c in col_order if c in metrics_df.columns]
    metrics_df = metrics_df[col_order]

    out_path = os.path.join(OUTPUT_DIR, "fund_metrics.csv")
    metrics_df.to_csv(out_path, index=False)

    # ── Summary report ───────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("  CALCULATION COMPLETE")
    print("=" * 60)
    print(f"\n  Funds processed      : {len(metrics_df)}")
    print(f"  Metrics per fund     : {len(metrics_df.columns)}")
    print(f"  Sharpe range         : {metrics_df['sharpe_ratio'].min():.3f} → {metrics_df['sharpe_ratio'].max():.3f}")
    print(f"  CAGR 3Y range        : {metrics_df['cagr_3y'].min():.2f}% → {metrics_df['cagr_3y'].max():.2f}%")
    print(f"  Max Drawdown range   : {metrics_df['max_drawdown_pct'].min():.2f}% → {metrics_df['max_drawdown_pct'].max():.2f}%")
    print(f"\n  Output files:")
    print(f"    → data/output/fund_metrics.csv")
    print(f"    → data/output/rolling_returns.csv")
    print("=" * 60)

    return metrics_df


# ─────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    result = run_all_calculations()
    print(f"\nPreview of fund_metrics.csv:")
    print(result[["fund_name", "cagr_3y", "sharpe_ratio",
                  "max_drawdown_pct", "composite_score", "universe_rank"]].to_string())

  MUTUAL FUND DASHBOARD — METRICS CALCULATION
Output folder ready: C:\Users\krish\OneDrive\Documents\Programming\Mutual_Fund_Dashboard\data\output/

[Loading clean files...]
  ✓ nav_history_clean.csv                    (22667, 4)
  ✓ nav_snapshots_clean.csv                  (13, 9)
  ✓ scheme_metadata_clean.csv                (14, 8)
  ✓ scheme_profiles_clean.csv                (13, 10)
  ✓ benchmark_data_clean.csv                 (1812, 9)
  ✓ category_reference_clean.csv             (41, 4)
  ✓ aum_amfi_monthly_clean.csv               (61, 11)

  Funds to process: 13

[Step 1] Calculating core metrics from NAV history...
  ✓ HDFC Top 100 Fund - Growth                         Sharpe: 0.3375  CAGR 3Y: 14.9976%
  ✓ Mirae Asset Large Cap Fund - Growth                Sharpe: 0.4205  CAGR 3Y: 12.4601%
  ✓ Kotak Equity Hybrid Fund - Growth                  Sharpe: 0.8495  CAGR 3Y: 23.2584%
  ✓ SBI Small Cap Fund - Growth                        Sharpe: 0.3163  CAGR 3Y: 10.7596%
  ✓ Axis Blue

In [7]:
BASE_DIR   = os.getcwd()
print(BASE_DIR)

C:\Users\krish\OneDrive\Documents\Programming\Mutual_Fund_Dashboard
